# Customer Segmentation & Retention Analysis

This notebook demonstrates how to analyze customer churn, segment users into actionable personas using **RFM (Recency, Frequency, Monetary)** proxies, and build predictive models to forecast who is likely to leave.

### Objectives:
1. **Exploratory Data Analysis (EDA)**: Understand customer distributions and patterns.
2. **Customer Segmentation**: Use K-Means clustering to discover customer profiles based on tenure, spend, and services.
3. **Churn Prediction**: Train and evaluate machine learning models (Logistic Regression, Random Forest, XGBoost) to predict customer churn.
4. **Business Interpretation**: Build strategy recommendations for each segment.

## 1. Setup & Import Libraries

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Set plots to display inline
%matplotlib inline
sns.set_style('whitegrid')

# Custom imports from our project source
import sys
sys.path.append('..')
from src.data_loader import load_and_clean_data
from src.features import engineer_features, preprocess_and_encode, scale_features
from src.segmentation import perform_segmentation
from src.models import train_and_evaluate_models

## 2. Load & Clean Data

In [ ]:
# Load the cleaned dataset
df = load_and_clean_data()
print(f"Dataset columns: {df.columns.tolist()}")
df.head()

## 3. Exploratory Data Analysis (EDA)

Let's explore the distribution of churn and key relationship characteristics.

In [ ]:
# 1. Overall Churn Rate
churn_counts = df['Churn'].value_counts(normalize=True) * 100
print(f"Churn Rates:\n{churn_counts}")

plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='Churn')
plt.title('Overall Customer Churn Count (0 = No, 1 = Yes)')
plt.ylabel('Number of Customers')
plt.show()

In [ ]:
# 2. Churn rate by contract type
plt.figure(figsize=(8, 5))
sns.countplot(data=df, x='Contract', hue='Churn')
plt.title('Churn by Contract Type')
plt.ylabel('Customer Count')
plt.show()

print("Month-to-month contracts see extremely high churn rates compared to 1 or 2-year terms.")

In [ ]:
# 3. Monthly charges and Tenure distribution relative to Churn
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.kdeplot(data=df, x='tenure', hue='Churn', fill=True, ax=axes[0])
axes[0].set_title('Tenure Distribution by Churn')
axes[0].set_xlabel('Tenure (months)')

sns.kdeplot(data=df, x='MonthlyCharges', hue='Churn', fill=True, ax=axes[1])
axes[1].set_title('Monthly Charges Distribution by Churn')
axes[1].set_xlabel('Monthly Charges ($)')

plt.tight_layout()
plt.show()

## 4. Customer Segmentation (RFM Clustering)

We adapt standard RFM principles to our subscription metrics:
- **Recency/Loyalty**: `tenure` (how long the customer has stayed)
- **Frequency**: `NumServices` (number of services they subscribe to)
- **Monetary**: `MonthlyCharges` (monthly expenditure)

We fit a **K-Means Clustering** model with $K=4$ clusters to identify key customer personas.

In [ ]:
# Perform segmentation and print centroid descriptions
df_segmented = perform_segmentation(df, n_clusters=4)

# Calculate cluster characteristics
cluster_summary = df_segmented.groupby('CustomerPersona')[['tenure', 'NumServices', 'MonthlyCharges', 'Churn']].mean()
cluster_sizes = df_segmented['CustomerPersona'].value_counts()
cluster_summary['Segment Size'] = cluster_sizes

print("\nCluster Characteristics Summary:")
cluster_summary

In [ ]:
# Visualize Clusters
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df_segmented, x='tenure', y='MonthlyCharges', hue='CustomerPersona', palette='Set2', alpha=0.7)
plt.title('Customer Segments: Tenure vs Monthly Charges')
plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.legend(title='Customer Persona')
plt.show()

## 5. Churn Prediction (Classification Modeling)

We train Logistic Regression, Random Forest, and XGBoost on our preprocessed features. 
We handle the class imbalance using balanced weights and compare their performance using **F1-Score** and **ROC-AUC**.

In [ ]:
results, feature_cols = train_and_evaluate_models(df)

In [ ]:
# Print summary tables
metrics_data = []
for name, metrics in results.items():
    metrics_data.append({
        "Model": name,
        "Accuracy": metrics["Accuracy"],
        "Precision": metrics["Precision"],
        "Recall": metrics["Recall"],
        "F1-Score": metrics["F1-Score"],
        "ROC-AUC": metrics["ROC-AUC"]
    })
metrics_df = pd.DataFrame(metrics_data)
print(metrics_df.to_string(index=False))

## 6. Feature Importance

Let's identify the most important factors driving customer churn by examining feature importances.

In [ ]:
import pickle
# Load best model
with open('../models/best_churn_model.pkl', 'rb') as f:
    model = pickle.load(f)
    
# Get feature importances / coefficients
if hasattr(model, 'feature_importances_'):
    importances = model.feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(10)
    title = 'Top 10 Feature Importances (Tree Model)'
elif hasattr(model, 'coef_'):
    importances = np.abs(model.coef_[0])
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(10)
    title = 'Top 10 Feature Coefficients (Logistic Regression)'
else:
    feat_imp = None

if feat_imp is not None:
    plt.figure(figsize=(10, 6))
    sns.barplot(x=feat_imp.values, y=feat_imp.index, palette='viridis')
    plt.title(title)
    plt.xlabel('Importance Value')
    plt.ylabel('Features')
    plt.show()
else:
    print("Could not extract feature importances from this model type.")

## 7. Business & Actionable Insights

Based on our segmentation and models, here are the tactical plays for the business:
1. **At-Risk Casuals (High Churn Rate ~50-60%)**:
   * *Characteristics*: Low tenure, Month-to-month contracts, single service, moderate spend.
   * *Strategy*: Early onboarding offers. Run campaigns encouraging a 1-year contract transition by bundling services (e.g., adding Online Backup or Security at a discounted rate).
2. **High-Value VIPs (Low Churn Rate ~10%)**:
   * *Characteristics*: Long tenure, multi-service, Fiber Optic, high MonthlyCharges.
   * *Strategy*: Priority customer support. Provide early access to new features. Proactive check-ins before contract renewal.
3. **Budget Loyalists (Low Churn Rate ~5%)**:
   * *Characteristics*: Long tenure, low spend, DSL/No internet, long-term contracts.
   * *Strategy*: Keep them happy with low-cost retention rewards. Do not upsell too aggressively as they are price-sensitive.
4. **Standard/Mid-Tier (Moderate Churn Rate ~20%)**:
   * *Characteristics*: Balanced spending and tenure.
   * *Strategy*: Cross-sell targeted value additions (e.g., streaming services or device protection).